> 在线讲义：[第 9 章 · 知识隔离与 π0.5：别让小脑把大脑的常识冲掉](https://wikieden.github.io/vla-from-scratch/lessons/09-knowledge-insulation.html) · 本书《VLA 从零到 π0》

# 第 9 章 · 知识隔离与 π0.5：别让小脑把大脑的常识冲掉

配套讲义：仓库里 `vla-deep-dive/notes/lessons/09-knowledge-insulation.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `grads(..., stop_gradient=False)` | 随机小脑 w_a = 0.3：大脑收到梯度 −1.08 |
| 第 2 步 | `grads(..., stop_gradient=True)` | 大脑梯度 0，小脑梯度不变（−3.60） |
| 第 3 步 | `PI05`、`KI_SPEEDUP` | 280k + 80k 步；目标场景 2.4%；π0 多 7.5 倍步数 |

梯度用一个两参数的玩具模型手算：大脑参数 w_b 产生 K/V，小脑参数 w_a 读它出动作。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
PI05 = {"预训练步数": 280_000, "后训练步数": 80_000, "非目标场景占比": 0.976}
KI_SPEEDUP = 7.5  # π0 要多 7.5 倍训练步数（KI 图 6）

**函数 `grads`**：前向：kv = w_b·x（大脑写白板），action = w_a·kv（小脑读白板）；损失 (action − target)²。

In [ ]:
def grads(w_b, w_a, x, target, stop_gradient):
    """前向：kv = w_b·x（大脑写白板），action = w_a·kv（小脑读白板）；损失 (action − target)²。

    返回 (∂L/∂w_b, ∂L/∂w_a)。stop_gradient=True 时，kv 在反向里当常数，梯度传不回大脑。
    """
    kv = w_b * x
    err = w_a * kv - target
    g_a = 2 * err * kv
    g_b = 0.0 if stop_gradient else 2 * err * w_a * x
    return g_b, g_a

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 一起训练：小脑的误差顺着 attention 流进大脑")
g_b, g_a = grads(w_b=1.0, w_a=0.3, x=2.0, target=1.5, stop_gradient=False)
print(f"  随机小脑 w_a = 0.3：大脑收到梯度 {g_b:+.2f}，小脑 {g_a:+.2f}")

In [ ]:
print("\n第二步 stop-gradient：前向一样，反向停在 sg")
g_b, g_a = grads(w_b=1.0, w_a=0.3, x=2.0, target=1.5, stop_gradient=True)
print(f"  加 sg：大脑收到梯度 {g_b:+.2f}（大脑另由 FAST 词和图文数据教），小脑 {g_a:+.2f}")
print(f"  π0 要多 {KI_SPEEDUP} 倍训练步数才追上")

In [ ]:
print("\n第三步 π0.5 的配方")
print(f"  预训练 {PI05['预训练步数']:,} 步 + 后训练 {PI05['后训练步数']:,} 步")
print(f"  {PI05['非目标场景占比']:.1%} 的预训练样本不是目标场景，目标场景只占 {1 - PI05['非目标场景占比']:.1%}")

## 3. 代码里才看得到的

1. `grads` 是手算的链式法则：大脑的梯度 = 误差 × w_a × x。小脑越"乱"（误差越大），流进大脑的梯度越大，这就是第 1 步说的训练刚开始伤害最大。
2. 加 sg 以后小脑自己的梯度一点没变：stop-gradient 只切断往大脑那一支，不影响小脑学习。

## 4. 自己改一改

- 把 w_a 改成接近"学好了"的值（让误差接近 0），看流进大脑的梯度还有多大。
- 改 `PI05` 的比例，算一算目标场景占 10% 时其余数据占多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 实习生干砸，老员工挨批改坏习惯；冻住老员工，又学不会新活。
> 知识隔离：小脑只读不改，大脑另有老师（FAST 词 + 图文数据）。
> π0.5：九成多是别处经验，先念词后接小脑，先写子任务再出动作。